# LangChain Agents — Beginner-Friendly Hands-On

## Tool Calling, ReAct Loop, Custom Tools, Tavily Search & Structured Responses

### What we will build

A simple **Research Agent** that can:

1. Answer normal questions using an LLM.
2. Use a calculator tool when a calculation is needed.
3. Search the web using Tavily when current information is needed.
4. Automatically choose the correct tool.
5. Return the final answer in a simple structured format.

### Learning flow

**LLM → Tool → Tool Calling → Agent → ReAct Loop → Custom Tools → Tavily → Structured Response**

> **Important:** We will learn the idea first and then write the code. Every important code section has comments explaining *what* we are doing and *why*.

## 1. What is an Agent?

A normal LLM application looks like:

```text
User → LLM → Answer
```

An **agent** can decide whether it needs to use a tool:

```text
User
  ↓
Agent / LLM
  ↓
Do I need a tool?
  ├── No  → Final Answer
  │
  └── Yes → Call Tool
              ↓
          Tool Result
              ↓
             LLM
              ↓
        Final Answer
```

### Simple analogy

- **LLM = Brain**
- **Tools = Hands**
- **Agent = Brain deciding when to use the hands**

We will start with a normal LLM and then gradually add these capabilities.

## 2. Install the Required Packages

Run this cell once.

### Packages used

- `langchain` → agent framework
- `langchain-google-genai` → connect LangChain to Gemini
- `langchain-tavily` → Tavily web-search tool
- `tavily-python` → Tavily dependency
- `python-dotenv` → load API keys from a `.env` file
- `pydantic` → define the structure of our final response

In [ ]:
# Run this cell once if the packages are not installed.
# If you are using Google Colab/Jupyter, you can uncomment and run it.

%pip install -U langchain langchain-google-genai langchain-tavily tavily-python python-dotenv pydantic

## 3. API Keys

For this notebook we use:

- **Google Gemini** as the LLM
- **Tavily** for web search

Create a `.env` file in the same folder as this notebook:

```text
GOOGLE_API_KEY=your_google_api_key
TAVILY_API_KEY=your_tavily_api_key
```

### Why do we need API keys?

The Python code needs permission to communicate with these external services.

> Never hard-code real API keys directly into a notebook that you plan to share publicly.

In [2]:
# Load the API keys from the .env file

from dotenv import load_dotenv

load_dotenv()

print("API keys loaded from .env")

API keys loaded from .env


## 4. Import the Gemini Chat Model

First, let's use Gemini just like a normal LLM.

At this stage there is **no agent and no tool**.

In [ ]:
from langchain.chat_models import init_chat_model


# temperature=0 makes the output more deterministic,
# which is useful for a classroom demonstration.
model = init_chat_model(
    "openai/gpt-oss-20b",
    model_provider = "openrouter",
    temperature=0
)

print("Open AI model created successfully.")

Open AI model created successfully.


## 5. Test the LLM

Let's ask a simple question.

This is the basic pattern:

```text
Question → LLM → Answer
```

In [6]:
# Ask the model a simple question

response = model.invoke("What is an AI agent?")

# .content contains the text generated by the model
print(response.content)

### Short answer  
An **AI agent** is a computer program (or network of programs) that can perceive its environment, reason about that perception, and take actions to achieve goals—usually while improving its own performance over time.

---

## 1. Core Concepts

| Concept | What it means for an agent | Why it matters |
|--------|---------------------------|----------------|
| **Perception / Sensing** | Reads input data (images, text, sensor streams, market feeds, etc.). | The agent’s world‑view; without it, it cannot decide. |
| **State Representation** | Internally encodes the relevant information gathered. | Must be abstract enough to allow reasoning and compact enough for computation. |
| **Reasoning / Decision‑Making** | Calculates a plan or policy based on the state. | Determines *what* to do next. |
| **Actuation / Output** | Sends commands to an environment (e.g., a robot wheel, a website request, a talk). | The means by which the agent influences the world. |
| **Learning / Ada

# Part A — Create Our First Tool

## 6. What is a Tool?

A **tool is a Python function that an agent is allowed to use**.

For example, we can create a function that multiplies two numbers.

The important difference is:

```text
Normal Python function
        ↓
Python program decides when to call it
```

With an agent:

```text
User question
      ↓
LLM decides whether the tool is useful
      ↓
Tool is called
```

In [39]:
from langchain.tools import tool

@tool
def multiply(a: float, b: int) -> float:
    """Multiply two numbers together."""

    # This is the actual Python work done by the tool.
    return a * b

## 7. Why did we use `@tool`?

The `@tool` decorator tells LangChain:

> "Make this Python function available as an LLM tool."

### Why is the docstring important?

```python
"""Multiply two numbers together."""
```

The LLM uses the tool's name, description, and input information to understand **when the tool should be used**.

Good tool descriptions help the agent select the correct tool.

In [38]:
# We can still call the tool directly.
# This is useful for understanding that a tool is still a Python function.

result = multiply.invoke({
    "a": 10.8,
    "b": 20
})

print("Result:", result)

Result: 216.0


Expected output:

```text
Result: 200
```

Notice:

**We called the tool ourselves.**

Next, we will let the **agent decide when to call it**.

# Part B — Build Our First Agent

## 8. Create an Agent with `create_agent`

LangChain provides `create_agent()` to create an agent.

We give it two things:

1. `model` → the LLM that makes decisions
2. `tools` → the tools the agent is allowed to use

So:

```text
Gemini Model + Tools
          ↓
        Agent
```

In [14]:
from langchain.agents import create_agent

# Create an agent and give it our multiply tool.
agent = create_agent(
    model=model,
    tools=[multiply]
)

print("Agent created successfully.")

Agent created successfully.


## 9. Ask the Agent to Calculate

Now we don't call `multiply()` ourselves.

Instead, we ask the **agent** a question.

The agent can decide:

> "This is a multiplication question, so I should use the multiply tool."

In [16]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 25 multiplied by 12?"
        }
    ]
})

# The agent keeps its conversation in the "messages" state.
# The last message normally contains the final response.
print(result["messages"])
print(result["messages"][-1].content)

[HumanMessage(content='What is 25 multiplied by 12?', additional_kwargs={}, response_metadata={}, id='b29c1950-153f-4f07-9993-b2edff808dbb'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks "What is 25 multiplied by 12?" I should use the function multiply.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user asks "What is 25 multiplied by 12?" I should use the function multiply.'}]}, response_metadata={'model_name': 'openai/gpt-oss-20b', 'id': 'gen-1788427583-rLgzPIXbiRH9ezEh6Pg0', 'created': 1788427583, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 1.001e-05, 'cost_details': {'upstream_inference_completions_cost': 6.44e-06, 'upstream_inference_prompt_cost': 3.57e-06, 'upstream_inference_cost': 1.001e-05}}, id='lc_run--01a06697-0dd7-7b13-b2ea-c50b89bad0ba-0', tool_calls=[{'name': 'multiply', 'args': {'a': 25, 'b': 12}, 'id': 'chatcmpl-too

## 10. What Happened Behind the Scenes?

Conceptually, the agent performed:

```text
User
 ↓
"What is 25 × 12?"
 ↓
LLM
 ↓
"I should use the multiply tool"
 ↓
multiply(25, 12)
 ↓
300
 ↓
LLM
 ↓
Final Answer
```

This is **tool calling**.

The LLM does not directly execute Python code.

Instead, it requests a tool call, and the agent runtime executes the tool.

# Part C — Understand the ReAct Loop

## 11. What is ReAct?

ReAct means:

**Reason + Act**

The useful classroom mental model is:

```text
Reason → Act → Observe → Repeat
```

For example:

```text
User asks a question
       ↓
Agent decides what to do
       ↓
Act: call a tool
       ↓
Observe: receive tool result
       ↓
Decide what to do next
       ↓
Final Answer
```

We are using "reason" here as a **conceptual description of the decision process**. We should not expose private chain-of-thought to users.

## 12. ReAct Example

Suppose the user asks:

> "Add 20 and 30, then multiply the result by 5."

The agent can perform:

```text
User
 ↓
Add 20 + 30
 ↓
Observe: 50
 ↓
Multiply 50 × 5
 ↓
Observe: 250
 ↓
Final Answer: 250
```

This demonstrates why agents are useful: the next action can depend on the result of the previous action.

# Part D — Add Another Custom Tool

## 13. Create an Addition Tool

Let's give our agent another capability.

Now it can choose between:

- `add`
- `multiply`

In [37]:
@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""

    # Perform the addition and return the result.
    return a + b

In [18]:
# Give both tools to the agent.
agent = create_agent(
    model=model,
    tools=[add, multiply]
)

print("Agent now has 2 tools.")

Agent now has 2 tools.


## 14. Test the Addition Tool

Ask:

> What is 100 + 250?

The important part is that **we don't tell the agent which tool to use**.

The agent should select `add`.

In [19]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "What is 100 + 250?"
        }
    ]
})

print(result["messages"][-1].content)

The sum of 100 and 250 is **350**.


## 15. Test Multiple Tool Calls

Now try a question that requires two steps:

> Add 20 and 30, then multiply the result by 5.

The agent may follow this conceptual process:

```text
add(20, 30)
      ↓
     50
      ↓
multiply(50, 5)
      ↓
    250
```

This is our first simple multi-step agent example.

In [20]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Add 20 and 30, then multiply the result by 5."
        }
    ]
})

print(result["messages"][-1].content)

The result is **250**.


## 16. Inspect the Agent Messages

For learning purposes, let's look at the messages returned by the agent.

This helps us understand that an agent interaction can contain:

```text
User message
     ↓
AI tool call
     ↓
Tool result
     ↓
AI final answer
```

In [21]:
# Print each message so we can see the agent's execution state.

for i, message in enumerate(result["messages"], start=1):
    print(f"--- Message {i} ---")
    print("Type:", type(message).__name__)
    print(message)
    print()

--- Message 1 ---
Type: HumanMessage
content='Add 20 and 30, then multiply the result by 5.' additional_kwargs={} response_metadata={} id='93d3356a-e077-4015-9f74-e8e82befb6a7'

--- Message 2 ---
Type: AIMessage
content='' additional_kwargs={'reasoning_content': 'We need to add 20 and 30, then multiply by 5. Use functions. First call add. Then multiply.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'We need to add 20 and 30, then multiply by 5. Use functions. First call add. Then multiply.'}]} response_metadata={'model_name': 'openai/gpt-oss-20b', 'id': 'gen-1788428621-fH7puOyH2JNNEhXs1dZt', 'created': 1788428621, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 8.48e-06, 'cost_details': {'upstream_inference_completions_cost': 5.3e-06, 'upstream_inference_prompt_cost': 3.18e-06, 'upstream_inference_cost': 8.48e-06}} id='lc_run--01a066a7-111a-7090-a2b7-8d1469cfed86-0' tool_

### Looking for tool calls

AI messages can contain a `tool_calls` attribute.

Let's inspect it when available.

In [22]:
for message in result["messages"]:
    if hasattr(message, "tool_calls") and message.tool_calls:
        print("Tool calls:")
        print(message.tool_calls)
        print()

Tool calls:
[{'name': 'add', 'args': {'a': 20, 'b': 30}, 'id': 'call_7101E67FE8F04519A3A010A8', 'type': 'tool_call'}]

Tool calls:
[{'name': 'multiply', 'args': {'a': 50, 'b': 5}, 'id': 'call_3CEF0FAAE32D4DBA807819C0', 'type': 'tool_call'}]



# Part E — Add Tavily Web Search

## 17. Why Do We Need Web Search?

Ask the LLM:

> "What is the latest news about AI?"

An LLM may not have live web access in the way a search engine does.

So we give the agent a **web-search tool**.

The architecture becomes:

```text
User
 ↓
Agent
 ↓
LLM
 ↓
Tavily Search
 ↓
Search Results
 ↓
LLM
 ↓
Final Answer
```

In [36]:
from langchain_tavily import TavilySearch

# Create a Tavily search tool.
# max_results controls how many search results are returned.
search = TavilySearch(
    max_results=5
)

print("Tavily search tool created.")

Tavily search tool created.


## 18. Give Tavily to the Agent

Now our agent has three tools:

```text
1. add
2. multiply
3. web search
```

The agent can decide which one is appropriate.

In [28]:
tools = [
    add,
    multiply,
    search
]

agent = create_agent(
    model=model,
    tools=[search]
)

print("Agent created with 3 tools.")

Agent created with 3 tools.


## 19. Test Web Search

Ask a question that clearly needs current information.

For example:

> "Search the web for the latest information about AI agents."

The agent should recognize that web search is useful.

In [26]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Movie review of TOXIC, and when this movie was released?"
        }
    ]
})

print(result["messages"][-1].content)

**Movie Review – *Toxic: A Fairy Tale for Grown‑Ups* (2026)**  

| Category | Verdict |
|----------|---------|
| **Plot** | A sprawling gangster‑drama that follows the rise and fall of the dual‑role protagonist, Raya (Yash) and his son Ticket. The narrative is ambitious, weaving family drama, crime‑family politics, and a touch of mythic grandeur. |
| **Direction** | Geetu Mohandas delivers a visually lush, almost operatic style. The pacing is uneven – the first 15 minutes set a grand tone, but the middle act drags with repetitive set‑pieces. The climax, however, is a tightly choreographed payoff that re‑energises the story. |
| **Performances** | Yash is solid in both roles, but the chemistry with Kiara Advani and Nayanthara feels forced at times. Huma Qureshi’s cameo is a highlight, adding a layer of emotional depth. |
| **Cinematography & Production Design** | The film is a visual feast: sweeping desert landscapes, opulent palace interiors, and gritty underworld alleys are rendered w

# Part F — The Main Agent Demonstration

## 20. Combine Search + Calculation

Now we will ask the agent a question that requires **more than one tool**.

Example:

> "Search the web for India's current population and calculate 2% of it."

The agent needs to:

1. Search for current population information.
2. Read the search result.
3. Perform a calculation.
4. Give the final answer.

This is a very good example of the **ReAct-style loop**.

In [ ]:
question = (
    "Search the web for India's current population "
    "and calculate 2% of that population."
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ]
})


print(result["messages"][-1].content)

**India’s population (2024)**  
According to the most recent United Nations–based estimate published by *GlobalData* in 2024, India’s population is **1,450,935,791** people 【1†content】.

**2 % of that population**

\[
1,450,935,791 \times 0.02 \;=\; 29,018,715.82
\]

So 2 % of India’s 2024 population is roughly **29 million** people (29,018,716 when rounded to the nearest whole person).


In [31]:
print(result["messages"])

[HumanMessage(content="Search the web for India's current population and calculate 2% of that population.", additional_kwargs={}, response_metadata={}, id='9a58d65a-2135-4d44-8a7a-8cee497b243b'), AIMessage(content='', additional_kwargs={'reasoning_content': "We need to search web for India's current population. Use tavily_search. Probably need up-to-date. Use advanced search. Then calculate 2%. Let's do search.", 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': "We need to search web for India's current population. Use tavily_search. Probably need up-to-date. Use advanced search. Then calculate 2%. Let's do search."}]}, response_metadata={'model_name': 'openai/gpt-oss-20b', 'id': 'gen-1788429897-7SGOLAoqMmduT7XFD60b', 'created': 1788429897, 'object': 'chat.completion', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'openrouter', 'cost': 3.594e-05, 'cost_details': {'upstream_inference_completions_cost': 1.04e-05, 'upstream_inf

## 21. Understand This Execution

Conceptually:

```text
User
 ↓
Agent
 ↓
LLM
 ↓
Need current population
 ↓
Tavily Search
 ↓
Population information
 ↓
LLM
 ↓
Need calculation
 ↓
Calculator Tool
 ↓
Result
 ↓
LLM
 ↓
Final Answer
```

### The key idea

The agent chooses the **next action based on the current situation**.

That is what makes it different from a fixed chain.

# Part G — Tool Calling vs Agent

## 22. Important Difference

### Tool Calling

The LLM can request:

```text
"Please call the multiply tool with a=10 and b=20."
```

### Agent

The system can repeatedly decide:

```text
Should I answer?
Should I search?
Should I calculate?
Should I call another tool?
Am I finished?
```

So remember:

> **Tool calling is a capability. An agent is a system that uses tool calling to make decisions and complete tasks.**

# Part H — Response Formatting

## 23. Why Structured Responses?

So far, the agent returns normal text.

For example:

```text
The answer is 250.
```

But applications often need predictable data.

For example:

```json
{
    "answer": "250",
    "summary": "The calculation was completed.",
    "sources": []
}
```

This is easier for a Python program, API, or frontend to consume.

In [40]:
from pydantic import BaseModel, Field

class AgentResponse(BaseModel):
    # The main answer shown to the user.
    answer: str = Field(
        description="The final answer to the user's question."
    )

    # A short explanation/summary.
    summary: str = Field(
        description="A short summary of the result."
    )

    # URLs or source names used during research.
    sources: list[str] = Field(
        description="Important sources used to produce the answer."
    )

print("Response schema created.")

Response schema created.


## 24. Create an Agent with Structured Response

We can tell `create_agent()` that we want the final response to follow our Pydantic schema.

The important idea is:

```text
Agent
 ↓
Final result
 ↓
Follow AgentResponse schema
```

This is different from tool calling.

- **Tool calling** → the model requests an action.
- **Structured response** → the final answer follows a predictable format.

In [33]:
agent_structured = create_agent(
    model=model,
    tools=tools,
    response_format=AgentResponse
)

print("Structured-output agent created.")

Structured-output agent created.


## 25. Test Structured Output

Let's ask a simple research question.

In [34]:
result = agent_structured.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Search the web and explain what an AI agent is in simple words."
        }
    ]
})

# create_agent stores the structured final result in "structured_response".
response = result["structured_response"]

print("ANSWER:")
print(response.answer)

print("\nSUMMARY:")
print(response.summary)

print("\nSOURCES:")
for source in response.sources:
    print("-", source)

ANSWER:
An **AI agent** is a piece of software that can act on its own to reach a goal you give it. Think of it like a tiny robot you write code for, but instead of moving a wheel, it moves in a digital world—like reading an email, filling out a form, or booking a flight. The key points are:

* **Goal‑oriented** – you tell it what you want (e.g., “schedule a meeting”) and it figures out the steps needed.
* **Autonomous decision‑making** – it chooses the best actions to get there, rather than just following a fixed script. It can adapt if something changes.
* **Uses AI** – it relies on machine‑learning models (often large language models) to understand language, plan, and learn from past tasks.

> **Simple example:** If you ask your AI agent to *“Book a dinner reservation for Tuesday at a good Italian restaurant,”* the agent will look up restaurant options, check availability, pick the best one, and place the booking without you clicking anything.

In short, an AI agent is a proactive, 

# Part I — Final Mini Project

## 26. Build a Research & Calculator Agent

Our final agent has:

### Model

```text
Gemini
```

### Tools

```text
Tavily Search
Add
Multiply
```

### Response format

```text
AgentResponse
 ├── answer
 ├── summary
 └── sources
```

### Architecture

```text
                    USER
                      ↓
                    AGENT
                      ↓
                     LLM
                      ↓
          ┌───────────┼───────────┐
          ↓           ↓           ↓
       Search        Add       Multiply
          │           │           │
          └───────────┼───────────┘
                      ↓
                 Tool Result
                      ↓
                     LLM
                      ↓
              Structured Response
                      ↓
                    USER
```

In [41]:
# ============================================
# FINAL RESEARCH & CALCULATOR AGENT
# ============================================

# 1. Create the agent.
final_agent = create_agent(
    model=model,
    tools=[
        search,      # Search the web
        add,         # Add numbers
        multiply     # Multiply numbers
    ],
    response_format=AgentResponse
)

# 2. Ask the agent a question.
question = (
    "Search the web for the latest information about AI agents. "
    "Then give me a simple explanation."
)

result = final_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": question
        }
    ]
})

# 3. Get the structured final response.
response = result["structured_response"]

# 4. Display the response.
print("ANSWER")
print(response.answer)

print("\nSUMMARY")
print(response.summary)

print("\nSOURCES")
for source in response.sources:
    print("-", source)

ANSWER
**What are AI agents?**

AI agents are autonomous software programs that can perceive an environment, plan, reason, use tools, and take a sequence of actions to achieve a goal—everything from completing a customer‑service ticket to controlling a robot arm. Think of them as “self‑driven chatbots” that don’t just answer one prompt, but can read a situation, break it into steps, act, and then adjust their plan.

**Why the hype now?**

1. **Breakthroughs with large‑language models (LLMs)** – The latest LLMs (like GPT‑4.5, GPT‑5.6, Claude‑3) can understand instructions, generate code, and remember context for many turns. Researchers and companies (OpenAI, Anthropic, LangChain, and the open‑source AutoGPT family) are building “agentic” frameworks that chain multiple tool calls and memory steps.

2. **Real‑world ecology** – Publishers and enterprise blogs now report that big firms (e.g., Microsoft, Google, Tesla) and retail giants (Tesco) are deploying AI agents to automate routine wor

# Part J — Student Practice Exercises

## Challenge 1 — Calculator

Ask the agent:

```text
What is 125 multiplied by 48?
```

**Expected behavior:** use the `multiply` tool.

---

## Challenge 2 — Addition

Ask:

```text
What is 450 + 275?
```

**Expected behavior:** use the `add` tool.

---

## Challenge 3 — Multiple Tools

Ask:

```text
Add 100 and 200, then multiply the result by 5.
```

Expected conceptual flow:

```text
add
 ↓
300
 ↓
multiply
 ↓
1500
```

---

## Challenge 4 — Web Search

Ask:

```text
Search the web for the latest developments in generative AI.
```

**Expected behavior:** use Tavily.

---

## Challenge 5 — Search + Calculation

Ask:

```text
Search the web for India's current population
and calculate 5% of it.
```

The agent should decide that it needs:

```text
Tavily → population information
       ↓
Calculation → 5%
```

---

# Final Challenge

Build your own tool.

For example:

```python
@tool
def square(number: int) -> int:
    """Return the square of a number."""
    return number * number
```

Add it to the agent and test:

```text
What is the square of 25?
```

# 27. Final Summary

You have built an agent step by step.

## Step 1 — Normal LLM

```text
User → LLM → Answer
```

## Step 2 — Create a Tool

```python
@tool
def multiply(...):
    ...
```

## Step 3 — Create an Agent

```python
agent = create_agent(
    model=model,
    tools=[multiply]
)
```

## Step 4 — Add More Tools

```text
add
multiply
```

## Step 5 — Add Tavily

```text
web search
```

## Step 6 — Understand ReAct

```text
Reason → Act → Observe → Repeat
```

## Step 7 — Structured Response

```python
class AgentResponse(BaseModel):
    answer: str
    summary: str
    sources: list[str]
```

---

# The Mental Model to Remember

```text
LLM
 +
Tools
 +
Decision Making
 +
Loop
 =
Agent
```

And:

```text
Tool Calling → "Which tool should I call?"

Agent → "What should I do next to solve the user's task?"

ReAct → "Reason/decide → Act → Observe → Repeat"

Structured Output → "Give the final result in a predictable format."
```

# 28. Troubleshooting

## API key error

Check that `.env` contains:

```text
GOOGLE_API_KEY=...
TAVILY_API_KEY=...
```

Then restart the notebook kernel if necessary and run:

```python
from dotenv import load_dotenv
load_dotenv()
```

---

## Tavily error

Check that:

```python
search = TavilySearch(max_results=5)
```

was created successfully and that `TAVILY_API_KEY` is available.

---

## Import error

Make sure the packages are installed in the **same Python environment/kernel** that is running this notebook.

You can check the Python executable with:

```python
import sys
print(sys.executable)
```

Then rerun the installation cell in that environment.

---

## Important version note

LangChain's APIs evolve quickly. If an import or argument differs in a future release, check the installed package version before changing the code.

You can see the installed version with:

```python
import langchain
print(langchain.__version__)
```

## Challenge 1 — Calculator

Ask the agent:

```text
What is 125 multiplied by 48?
```

**Expected behavior:** use the `multiply` tool.

In [ ]:
# create a tool

@tool
def multiply(a: float, b: int) -> float:
    """Multiply two numbers together."""

    # This is the actual Python work done by the tool.
    return a * b

# create an agent
agent = create_agent(
    model=model,
    tools=[multiply]
)

# Ask the agent a question.

question = "What is 25 multiplied by 12?"

result = agent.invoke(
    {
    "messages": [
            {"role": "user", "content": question}
            ]
    }
)

print(result["messages"][-1].content)



25 multiplied by 12 is **300**.


## Challenge 2 — Addition

Ask:

```text
What is 450 + 275?
```

In [46]:
# create a tool

@tool
def add(a: float, b: int) -> float:
    """Add two numbers together."""

    # This is the actual Python work done by the tool.
    return a + b

# create an agent
agent = create_agent(
    model=model,
    tools=[add]
)

# Ask the agent a question.

question = "What is 25 plus 12?"

result = agent.invoke(
    {
    "messages": [
            {"role": "user", "content": question}
            ]
    }
)

print(result["messages"][-1].content)



The sum of 25 and 12 is **37**.


## Challenge 3 — Multiple Tools

Ask:

```text
Add 100 and 200, then multiply the result by 5.
```

Expected conceptual flow:

```text
add
 ↓
300
 ↓
multiply
 ↓
1500
```

---

In [47]:
# create an agent
agent = create_agent(
    model=model,
    tools=[add,multiply]
)

# Ask the agent a question.

question = "What is 25 plus 12 and then multiplied by 2?"

result = agent.invoke(
    {
    "messages": [
            {"role": "user", "content": question}
            ]
    }
)

print(result["messages"][-1].content)



The result is **74**.


## Challenge 4 — Web Search

Ask:

```text
Search the web for the latest developments in generative AI.
```

**Expected behavior:** use Tavily.

---

In [54]:
# create an agent
agent = create_agent(
    model=model,
    tools=[search]
)

# Ask the agent a question.

question = "Search the web for the latest developments in generative AI"

result = agent.invoke(
    {
    "messages": [
            {"role": "user", "content": question}
            ]
    }
)

print(result["messages"][-1].content)

**What’s happening now?**  
The generative‑AI landscape has entered its “first‑hand‑experience” stage: models are no longer just labs‑bench experiments, they’re being put into production, offered as API products, and used for a growing list of mission‑critical workflows. Below is a snapshot of the most recent, high‑profile developments (late 2023 – mid 2024, with a look ahead to the next few months). All items are backed by the most recent primary announcements or peer‑reviewed reports and are not older than 90 days.

| Date & Source | Development | What it means | Key take‑away |
|---|---|---|---|
| **June 21 2024** – Anthropic | **Claude 3.5 Sonnet** (first in the 3.5 family) | 2 × the speed of Claude 3 Opus with 200 K‑token context, new “computer‑use” capability in public beta | The first frontier‑AI model that can *manually* navigate a UI – a step toward “agentic” AI that does more than answer prompts. <sup>1</sup> |
| **July 2024** – Google | **Gemini 1.5 Flash** (internal: “Gemin

## Challenge 5 — Search + Calculation

Ask:

```text
Search the web for India's current population
and calculate 5% of it.
```

The agent should decide that it needs:

```text
Tavily → population information
       ↓
Calculation → 5%

In [51]:
# create an agent
agent = create_agent(
    model=model,
    tools=[add,multiply,search]
)

# Ask the agent a question.

question = "Search the web for India's current population and calculate 5% of it."

result = agent.invoke(
    {
    "messages": [
            {"role": "user", "content": question}
            ]
    }
)

print(result["messages"][-1].content)

**India’s current population (2024 estimate)**  
- **1,450,935,791** people (source: *Wikipedia – Demographics of India*; see the 2024 population figure in the table).

**5 % of that population**

\[
\text{5 %} = 0.05 \times 1{,}450{,}935{,}791 \approx 72{,}546{,}789.55
\]

Rounded to the nearest whole person:

\[
\boxed{72{,}546{,}790}
\]

So, 5 % of India’s current population is about **72.5 million people**.


# Final Challenge

Build your own tool.

For example:

```python
@tool
def square(number: int) -> int:
    """Return the square of a number."""
    return number * number
```

Add it to the agent and test:

```text
What is the square of 25?

In [53]:
@tool
def square(number: int) -> int:
    """Return the square of a number."""
    return number * number

agent = create_agent(
    model=model,
    tools=[square]
)

question = "What is the square of 15?"

result =agent.invoke(
    {
    "messages": [
            {"role": "user", "content": question}
            ]
    }
)

print(result["messages"][-1].content)

The square of 15 is **225**.
